In [ ]:
from huggingface_hub import login
login()

In [ ]:
# KYROFORM AI PROTOTYPE: Gut-Host PPI Prediction HGNN
#ignore this first snippet gated model chalena hero vayera try gareko

!pip install torch torch-geometric biopython transformers


import torch
from torch_geometric.data import HeteroData
from torch_geometric.nn import RGCNConv
from torch.nn import Linear, BCEWithLogitsLoss
import torch.nn.functional as F
from Bio import SeqIO
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

# ESM-2 650M
print("Loading ESM-2 650M model...")
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
model = AutoModel.from_pretrained("facebook/esm2_t33_650M_UR50D")
model.eval()
if torch.cuda.is_available():
    model.cuda()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.0 MB/s eta 0:00:00
Loading ESM-2 650M model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Parse FASTA
fasta_path = "all_proteins.fasta"
sequences = {}
for record in SeqIO.parse(fasta_path, "fasta"):
    header = record.id
    if "|" in header:
        uniprot_id = header.split("|")[1]
    else:
        uniprot_id = header.split()[0].replace(">", "")
    sequences[uniprot_id] = str(record.seq).upper()  # ESM needs uppercase

print(f"Loaded {len(sequences)} sequences")

Loaded 1138 sequences


In [ ]:
# Embeddings (batch, mean pool)
embeds = {}
batch_size = 4  # Safe for 650M on Colab GPU
ids = list(sequences.keys())
for i in range(0, len(ids), batch_size):
    batch_ids = ids[i:i+batch_size]
    batch_seqs = [sequences[pid] for pid in batch_ids]
    inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        batch_emb = model(**inputs).last_hidden_state.mean(dim=1).cpu().numpy()
    for pid, emb in zip(batch_ids, batch_emb):
        embeds[pid] = emb

print(f"Embeddings done: {len(embeds)} proteins, dim {embeds[next(iter(embeds))].shape[0]}")


In [ ]:
from tqdm import tqdm  # Add this import if missing

embeds = {}
batch_size = 8
use_gpu = torch.cuda.is_available()
print(f"GPU available: {use_gpu} (if No, change Runtime → T4 GPU)")

total_batches = len(ids) // batch_size + (1 if len(ids) % batch_size else 0)

for i in tqdm(range(0, len(ids), batch_size), total=total_batches, desc="Generating ESM-2 Embeddings"):
    batch_ids = ids[i:i+batch_size]
    batch_seqs = [sequences[pid] for pid in batch_ids]

    inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024)

    if use_gpu:
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        batch_emb = outputs.last_hidden_state.mean(dim=1)  # Mean pool

    if use_gpu:
        batch_emb = batch_emb.cpu()

    batch_emb = batch_emb.numpy()

    for pid, emb in zip(batch_ids, batch_emb):
        embeds[pid] = emb

print(f"Embeddings done: {len(embeds)} proteins, dim {embeds[next(iter(embeds))].shape[0]}")

In [ ]:
# Load edges
df_edges = pd.read_csv("training_edges_with_labels.csv")

# Node mapping
human_prots = df_edges['human'].unique().tolist()
bact_prots = df_edges['bacterial'].unique().tolist()
h2idx = {p: i for i, p in enumerate(human_prots)}
b2idx = {p: i for i, p in enumerate(bact_prots)}

In [ ]:
# HeteroData
data = HeteroData()
data['human'].x = torch.tensor(np.stack([embeds.get(p, np.zeros(1280)) for p in human_prots]), dtype=torch.float)  # 1280 dim for 650M
data['bacterial'].x = torch.tensor(np.stack([embeds.get(p, np.zeros(1280)) for p in bact_prots]), dtype=torch.float)

src = torch.tensor([h2idx[h] for h in df_edges['human']])
dst = torch.tensor([b2idx[b] for b in df_edges['bacterial']])
data['human', 'interacts', 'bacterial'].edge_index = torch.stack([src, dst])
data['human', 'interacts', 'bacterial'].edge_label = torch.tensor(df_edges['label'].values, dtype=torch.float)

# Split
mask = torch.randperm(data['human', 'interacts', 'bacterial'].edge_label.size(0))
train_mask = mask[:int(0.8*len(mask))]
val_mask = mask[int(0.8*len(mask)):]

NameError: name 'embeds' is not defined

In [ ]:
# After embeddings are ready (embeds dict with ~1143 entries)

# Filter proteins to those with embeddings
human_prots = [p for p in df_edges['human'].unique() if p in embeds]
bact_prots = [p for p in df_edges['bacterial'].unique() if p in embeds]

print(f"Proteins with embeddings: {len(human_prots)} human, {len(bact_prots)} bacterial")

# Re-index edges to only valid proteins
valid_edges = df_edges[df_edges['human'].isin(embeds) & df_edges['bacterial'].isin(embeds)]

if len(valid_edges) < len(df_edges) * 0.9:
    print(f"Warning: Lost {len(df_edges) - len(valid_edges)} edges due to missing sequences")
else:
    print("All edges preserved")

# Node mapping
h2idx = {p: i for i, p in enumerate(human_prots)}
b2idx = {p: i for i, p in enumerate(bact_prots)}

# HeteroData
data = HeteroData()

# Node features (only valid proteins)
data['human'].x = torch.tensor(np.stack([embeds[p] for p in human_prots]), dtype=torch.float)
data['bacterial'].x = torch.tensor(np.stack([embeds[p] for p in bact_prots]), dtype=torch.float)

# Edges (filtered)
src = torch.tensor([h2idx[h] for h in valid_edges['human']])
dst = torch.tensor([b2idx[b] for b in valid_edges['bacterial']])
data['human', 'interacts', 'bacterial'].edge_index = torch.stack([src, dst])
data['human', 'interacts', 'bacterial'].edge_label = torch.tensor(valid_edges['label'].values, dtype=torch.float)

# Train/val split
num_edges = data['human', 'interacts', 'bacterial'].edge_label.size(0)
perm = torch.randperm(num_edges)
train_mask = perm[:int(0.8 * num_edges)]
val_mask = perm[int(0.8 * num_edges):]

print(f"Final graph: {data['human'].num_nodes} human nodes, {data['bacterial'].num_nodes} bacterial nodes")
print(f"{num_edges} edges for training")

NameError: name 'embeds' is not defined

In [ ]:
missing_human = [p for p in df_edges['human'] if p not in embeds]
missing_bact = [p for p in df_edges['bacterial'] if p not in embeds]

print(f"Missing embeddings for {len(missing_human)} human proteins (e.g., {missing_human[:10]})")
print(f"Missing embeddings for {len(missing_bact)} bacterial proteins (e.g., {missing_bact[:10]})")
print(f"Total edges affected: {df_edges[df_edges['human'].isin(missing_human) | df_edges['bacterial'].isin(missing_bact)].shape[0]}")

In [ ]:
# Diagnostic (optional, shows loss)
missing_h = set(df_edges['human']) - set(embeds.keys())
missing_b = set(df_edges['bacterial']) - set(embeds.keys())
print(f"Missing: {len(missing_h)} human, {len(missing_b)} bacterial → drop {df_edges[df_edges['human'].isin(missing_h) | df_edges['bacterial'].isin(missing_b)].shape[0]} edges")

# Filter edges to valid proteins
valid_df = df_edges[df_edges['human'].isin(embeds) & df_edges['bacterial'].isin(embeds)].reset_index(drop=True)
print(f"Kept {len(valid_df)} edges ({len(valid_df)/len(df_edges)*100:.1f}%) — excellent!")

# Unique valid proteins
human_prots = valid_df['human'].unique().tolist()
bact_prots = valid_df['bacterial'].unique().tolist()

h2idx = {p: i for i, p in enumerate(human_prots)}
b2idx = {p: i for i, p in enumerate(bact_prots)}

# HeteroData
data = HeteroData()
data['human'].x = torch.tensor(np.stack([embeds[p] for p in human_prots]), dtype=torch.float)
data['bacterial'].x = torch.tensor(np.stack([embeds[p] for p in bact_prots]), dtype=torch.float)

src = torch.tensor([h2idx[h] for h in valid_df['human']])
dst = torch.tensor([b2idx[b] for b in valid_df['bacterial']])
data['human', 'interacts', 'bacterial'].edge_index = torch.stack([src, dst])
data['human', 'interacts', 'bacterial'].edge_label = torch.tensor(valid_df['label'].values, dtype=torch.float)

# Split
num_edges = len(valid_df)
perm = torch.randperm(num_edges)
train_mask = perm[:int(0.8 * num_edges)]
val_mask = perm[int(0.8 * num_edges):]

print(f"Graph ready: {len(human_prots)} human + {len(bact_prots)} bacterial nodes, {num_edges} edges")

In [ ]:
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

class HeteroSAGE(torch.nn.Module):
    def __init__(self, hidden_channels=256):
        super().__init__()
        self.conv1_h = SAGEConv((-1, -1), hidden_channels)
        self.conv2_h = SAGEConv(hidden_channels, hidden_channels)
        self.conv1_b = SAGEConv((-1, -1), hidden_channels)
        self.conv2_b = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x_dict, edge_index_dict):
        edge_index = edge_index_dict[('human', 'interacts', 'bacterial')]
        rev_edge = edge_index.flip(0)  # Safe reverse for human incoming

        # Human nodes: receive from bacterial
        h = F.relu(self.conv1_h(x_dict['human'], rev_edge))
        h = F.relu(self.conv2_h(h, rev_edge))

        # Bacterial nodes: receive from human
        b = F.relu(self.conv1_b(x_dict['bacterial'], edge_index))
        b = F.relu(self.conv2_b(b, edge_index))

        return {'human': h, 'bacterial': b}

model = HeteroSAGE(hidden_channels=256)

if torch.cuda.is_available():
    model.cuda()
    data = data.cuda()

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
criterion = BCEWithLogitsLoss()

def decode(z_h, z_b):
    return (z_h * z_b).sum(dim=-1)

print("FINAL TRAINING STARTED — Kyroform AI coming online...")
best_val_auc = 0
for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()

    z = model(data.x_dict, data.edge_index_dict)

    # Positive
    pos_edge = data['human', 'interacts', 'bacterial'].edge_index
    pos_pred = decode(z['human'][pos_edge[0]], z['bacterial'][pos_edge[1]])

    # Negative (in-batch random)
    neg_num = pos_pred.size(0)
    neg_src = torch.randint(0, z['human'].size(0), (neg_num,), device=pos_pred.device)
    neg_dst = torch.randint(0, z['bacterial'].size(0), (neg_num,), device=pos_pred.device)
    neg_pred = decode(z['human'][neg_src], z['bacterial'][neg_dst])

    pred = torch.cat([pos_pred, neg_pred])
    label = torch.cat([torch.ones(pos_pred.size(0)), torch.zeros(neg_pred.size(0))]).to(pred.device)

    loss = criterion(pred, label)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0 or epoch == 200:
        model.eval()
        with torch.no_grad():
            z = model(data.x_dict, data.edge_index_dict)
            # Val positives
            val_pos_idx = val_mask
            val_pos_pred = decode(z['human'][pos_edge[0][val_pos_idx]], z['bacterial'][pos_edge[1][val_pos_idx]])
            # Val negatives
            val_neg_pred = decode(z['human'][torch.randint(0, z['human'].size(0), val_pos_idx.shape, device=val_pos_pred.device)],
                                  z['bacterial'][torch.randint(0, z['bacterial'].size(0), val_pos_idx.shape, device=val_pos_pred.device)])
            val_pred = torch.cat([val_pos_pred, val_neg_pred])
            val_label = torch.cat([torch.ones(val_pos_pred.size(0)), torch.zeros(val_neg_pred.size(0))]).to(val_pred.device)

            val_auc = roc_auc_score(val_label.cpu(), torch.sigmoid(val_pred).cpu())
            val_ap = average_precision_score(val_label.cpu(), torch.sigmoid(val_pred).cpu())
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                torch.save(model.state_dict(), "kyroform_best_model.pth")

        print(f'Epoch {epoch:03d} | Loss: {loss.item():.4f} | Val AUC: {val_auc:.4f} | Val AP: {val_ap:.4f} | Best AUC: {best_val_auc:.4f}')

print("\nKYROFORM AI TRAINING COMPLETE!")
print("Final model saved as 'kyroform_best_model.pth'")

In [ ]:
# Force CPU
device = torch.device('cpu')
print("Using CPU for training (somehow avoids CUDA crash)")

model = HeteroSAGE(hidden_channels=256)
model.to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
criterion = BCEWithLogitsLoss()

def decode(z_h, z_b):
    return (z_h * z_b).sum(dim=-1)

print("TRAINING ON CPU — stable and fast for prototype...")
for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()

    z = model(data.x_dict, data.edge_index_dict)

    pos_edge = data['human', 'interacts', 'bacterial'].edge_index
    pos_pred = decode(z['human'][pos_edge[0]], z['bacterial'][pos_edge[1]])

    neg_num = pos_pred.size(0)
    neg_src = torch.randint(0, z['human'].size(0), (neg_num,))
    neg_dst = torch.randint(0, z['bacterial'].size(0), (neg_num,))
    neg_pred = decode(z['human'][neg_src], z['bacterial'][neg_dst])

    pred = torch.cat([pos_pred, neg_pred])
    label = torch.cat([torch.ones(pos_pred.size(0)), torch.zeros(neg_pred.size(0))])

    loss = criterion(pred, label)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            z = model(data.x_dict, data.edge_index_dict)
            val_pos_pred = decode(z['human'][pos_edge[0][val_mask]], z['bacterial'][pos_edge[1][val_mask]])
            val_neg_src = torch.randint(0, z['human'].size(0), val_mask.shape)
            val_neg_dst = torch.randint(0, z['bacterial'].size(0), val_mask.shape)
            val_neg_pred = decode(z['human'][val_neg_src], z['bacterial'][val_neg_dst])
            val_pred = torch.cat([val_pos_pred, val_neg_pred])
            val_label = torch.cat([torch.ones(val_pos_pred.size(0)), torch.zeros(val_neg_pred.size(0))])
            val_auc = roc_auc_score(val_label.numpy(), torch.sigmoid(val_pred).numpy())
        print(f'Epoch {epoch:03d} | Loss: {loss.item():.4f} | Val AUC: {val_auc:.4f}')

torch.save(model.state_dict(), "kyroform_cpu_trained_model.pth")
print("\nFinally no crash!")

In [ ]:
import pickle

# Save embeddings
embed_path = "esm2_embeddings_1143_proteins.pkl"
with open(embed_path, 'wb') as f:
    pickle.dump(embeds, f)

print(f"Embeddings saved to?? {embed_path}")


In [ ]:
import pickle
import torch
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score # Added average_precision_score
import pandas as pd
import numpy as np

# Load emb
embed_path = "esm2_embeddings_1143_proteins.pkl"
with open(embed_path, 'rb') as f:
    embeds = pickle.load(f)

print(f"Loaded embfor {len(embeds)} ")

# Load edges
df_edges = pd.read_csv("training_edges_with_labels.csv")

# Filter to proteins with embeddings (safe)
valid_df = df_edges[df_edges['human'].isin(embeds) & df_edges['bacterial'].isin(embeds)].reset_index(drop=True)
print(f"Using {len(valid_df)} valid edges for training")

human_prots = valid_df['human'].unique().tolist()
bact_prots = valid_df['bacterial'].unique().tolist()

print(f"Number of human proteins: {len(human_prots)}")
print(f"Number of bacterial proteins: {len(bact_prots)}")

h2idx = {p: i for i, p in enumerate(human_prots)}
b2idx = {p: i for i, p in enumerate(bact_prots)}

# Build graph
data = HeteroData()
data['human'].x = torch.tensor(np.stack([embeds[p] for p in human_prots]), dtype=torch.float)
data['bacterial'].x = torch.tensor(np.stack([embeds[p] for p in bact_prots]), dtype=torch.float)

print(f"data['human'].x shape: {data['human'].x.shape}")
print(f"data['bacterial'].x shape: {data['bacterial'].x.shape}")

src = torch.tensor([h2idx[h] for h in valid_df['human']])
dst = torch.tensor([b2idx[b] for b in valid_df['bacterial']])
data['human', 'interacts', 'bacterial'].edge_index = torch.stack([src, dst])
data['human', 'interacts', 'bacterial'].edge_label = torch.tensor(valid_df['label'].values, dtype=torch.float)

perm = torch.randperm(len(valid_df))
train_mask = perm[:int(0.8 * len(valid_df))]
val_mask = perm[int(0.8 * len(valid_df)):]

# Model (same as final)
class HeteroSAGE(torch.nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.h_conv1 = SAGEConv((-1, -1), hidden)
        self.h_conv2 = SAGEConv(hidden, hidden)
        self.b_conv1 = SAGEConv((-1, -1), hidden)
        self.b_conv2 = SAGEConv(hidden, hidden)

    def forward(self, x_dict, edge_index_dict):
        edge = edge_index_dict[('human', 'interacts', 'bacterial')]
        rev = edge.flip(0) # Flips (src, dst) to (dst, src) -> (bacterial_idx, human_idx)

        # First layer
        # Human nodes receive from bacterial nodes
        h = F.relu(self.h_conv1((x_dict['human'], x_dict['bacterial']), rev))
        # Bacterial nodes receive from human nodes
        b = F.relu(self.b_conv1((x_dict['bacterial'], x_dict['human']), edge))

        # Second layer
        # Human nodes receive from bacterial nodes (using updated features h and b)
        h = F.relu(self.h_conv2((h, b), rev))
        # Bacterial nodes receive from human nodes (using updated features h and b)
        b = F.relu(self.b_conv2((b, h), edge))

        return {'human': h, 'bacterial': b}

model = HeteroSAGE()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = BCEWithLogitsLoss()

def decode(z_h, z_b):
    return (z_h * z_b).sum(dim=-1)

print("Training from loaded embeddings...")

best_val_auc = 0 # Track best validation AUC

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()
    z = model(data.x_dict, data.edge_index_dict)
    pos_edge = data['human', 'interacts', 'bacterial'].edge_index
    pos_pred = decode(z['human'][pos_edge[0]], z['bacterial'][pos_edge[1]])

    neg_num = pos_pred.size(0) # Use the same number of negatives as positives
    # Generate negative samples: random human-bacterial pairs
    neg_src = torch.randint(0, z['human'].size(0), (neg_num,))
    neg_dst = torch.randint(0, z['bacterial'].size(0), (neg_num,))
    neg_pred = decode(z['human'][neg_src], z['bacterial'][neg_dst])

    pred = torch.cat([pos_pred, neg_pred])
    label = torch.cat([torch.ones(pos_pred.size(0)), torch.zeros(neg_pred.size(0))])
    loss = criterion(pred, label)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0 or epoch == 200: # Check validation metrics every 20 epochs or at the end
        model.eval()
        with torch.no_grad():
            z = model(data.x_dict, data.edge_index_dict)
            # Validation positives
            val_pos = decode(z['human'][pos_edge[0][val_mask]], z['bacterial'][pos_edge[1][val_mask]])

            # Validation negatives (same number as validation positives)
            val_neg_num = val_pos.size(0)
            val_neg_src = torch.randint(0, z['human'].size(0), (val_neg_num,))
            val_neg_dst = torch.randint(0, z['bacterial'].size(0), (val_neg_num,))
            val_neg = decode(z['human'][val_neg_src], z['bacterial'][val_neg_dst])

            val_pred = torch.cat([val_pos, val_neg])
            val_label = torch.cat([torch.ones(val_pos.size(0)), torch.zeros(val_neg.size(0))])

            val_auc = roc_auc_score(val_label.numpy(), torch.sigmoid(val_pred).numpy())
            val_ap = average_precision_score(val_label.numpy(), torch.sigmoid(val_pred).numpy())

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                torch.save(model.state_dict(), "kyroform_final_model_from_saved_embeddings.pth") # Save the best model

        print(f'Epoch {epoch:03d} | Loss: {loss.item():.4f} | Val AUC: {val_auc:.4f} | Val AP: {val_ap:.4f} | Best AUC: {best_val_auc:.4f}')

print("Training completeeeeeeee")

In [ ]:
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

class HeteroSAGE(torch.nn.Module):
    def __init__(self, hidden_channels=256):
        super().__init__()
        self.h_conv1 = SAGEConv((-1, -1), hidden_channels)
        self.h_conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.b_conv1 = SAGEConv((-1, -1), hidden_channels)
        self.b_conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x_dict, edge_index_dict):
        edge = edge_index_dict[('human', 'interacts', 'bacterial')]

        # Bacterial → human (reverse edge)
        rev_edge = edge.flip(0)

        # Human update (receives from bacterial)
        h = F.relu(self.h_conv1(x_dict['human'], rev_edge))
        h = F.relu(self.h_conv2(h, rev_edge))

        # Bacterial update (receives from human)
        b = F.relu(self.b_conv1(x_dict['bacterial'], edge))
        b = F.relu(self.b_conv2(b, edge))

        return {'human': h, 'bacterial': b}

model = HeteroSAGE(hidden_channels=256)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
criterion = BCEWithLogitsLoss()

def decode(z_h, z_b):
    return (z_h * z_b).sum(dim=-1)

print("KYROFORM AI — FINAL TRAINING LAUNCH")
for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()

    z = model(data.x_dict, data.edge_index_dict)

    pos_edge = data['human', 'interacts', 'bacterial'].edge_index
    pos_pred = decode(z['human'][pos_edge[0]], z['bacterial'][pos_edge[1]])

    neg_src = torch.randint(0, z['human'].size(0), (pos_pred.size(0),))
    neg_dst = torch.randint(0, z['bacterial'].size(0), (pos_pred.size(0),))
    neg_pred = decode(z['human'][neg_src], z['bacterial'][neg_dst])

    pred = torch.cat([pos_pred, neg_pred])
    label = torch.cat([torch.ones(pos_pred.size(0)), torch.zeros(neg_pred.size(0))])

    loss = criterion(pred, label)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            z = model(data.x_dict, data.edge_index_dict)
            val_pos_pred = decode(z['human'][pos_edge[0][val_mask]], z['bacterial'][pos_edge[1][val_mask]])
            val_neg_pred = decode(z['human'][torch.randint(0, z['human'].size(0), (val_mask.size(0),))],
                                  z['bacterial'][torch.randint(0, z['bacterial'].size(0), (val_mask.size(0),))])
            val_pred = torch.cat([val_pos_pred, val_neg_pred])
            val_label = torch.cat([torch.ones(val_pos_pred.size(0)), torch.zeros(val_neg_pred.size(0))])
            val_auc = roc_auc_score(val_label.numpy(), torch.sigmoid(val_pred).numpy())
        print(f'Epoch {epoch:03d} | Loss: {loss.item():.4f} | Val AUC: {val_auc:.4f}')

torch.save(model.state_dict(), "kyroform_trained_model_2026.pth")
print("\nKYROFORM AI IS COMPLETE AND TRAINED.")

In [ ]:
# 1. Get unique proteins ONLY from the filtered edges you are actually using
human_prots = sorted(valid_df['human'].unique().tolist())
bact_prots = sorted(valid_df['bacterial'].unique().tolist())

# 2. Re-create mappings
h2idx = {p: i for i, p in enumerate(human_prots)}
b2idx = {p: i for i, p in enumerate(bact_prots)}

# 3. Build Node Features
data = HeteroData()
# Ensure these stacks match the human_prots/bact_prots list length exactly
data['human'].x = torch.tensor(np.stack([embeds[p] for p in human_prots]), dtype=torch.float)
data['bacterial'].x = torch.tensor(np.stack([embeds[p] for p in bact_prots]), dtype=torch.float)

# 4. Build Edge Index
# Use the new mappings to ensure indices match the feature matrix size
src = torch.tensor([h2idx[h] for h in valid_df['human']], dtype=torch.long)
dst = torch.tensor([b2idx[b] for b in valid_df['bacterial']], dtype=torch.long)
data['human', 'interacts', 'bacterial'].edge_index = torch.stack([src, dst])

# Debug check: This should now print (283) and (855) or similar based on your error
print(f"Human nodes: {data['human'].x.size(0)}, Max index in src: {src.max().item()}")
print(f"Bact nodes: {data['bacterial'].x.size(0)}, Max index in dst: {dst.max().item()}")

In [ ]:
from torch.nn import BCEWithLogitsLoss

# Initialize the model with the same hidden dimension as your notebook
model = HeteroSAGE(hidden_channels=256)

# Optimizer and Criterion
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
criterion = BCEWithLogitsLoss()

def decode(z_h, z_b):
    # Dot product between node embeddings to predict interaction
    return (z_h * z_b).sum(dim=-1)

In [ ]:
from sklearn.metrics import roc_auc_score

print("KYROFORM AI — FINAL TRAINING LAUNCH")

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()

    # Forward pass: Generate embeddings for all nodes
    z = model(data.x_dict, data.edge_index_dict)

    # Positive predictions (actual interactions)
    pos_edge = data['human', 'interacts', 'bacterial'].edge_index
    pos_pred = decode(z['human'][pos_edge[0]], z['bacterial'][pos_edge[1]])

    # Negative predictions (random pairs)
    neg_src = torch.randint(0, z['human'].size(0), (pos_pred.size(0),))
    neg_dst = torch.randint(0, z['bacterial'].size(0), (pos_pred.size(0),))
    neg_pred = decode(z['human'][neg_src], z['bacterial'][neg_dst])

    # Calculate Loss
    pred = torch.cat([pos_pred, neg_pred])
    label = torch.cat([torch.ones(pos_pred.size(0)), torch.zeros(neg_pred.size(0))])

    loss = criterion(pred, label)
    loss.backward()
    optimizer.step()

    # Log progress every 20 epochs
    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            # Use validation mask to calculate performance
            val_pos_pred = decode(z['human'][pos_edge[0][val_mask]], z['bacterial'][pos_edge[1][val_mask]])
            val_neg_pred = decode(z['human'][torch.randint(0, z['human'].size(0), (val_mask.size(0),))],
                                  z['bacterial'][torch.randint(0, z['bacterial'].size(0), (val_mask.size(0),))])

            val_pred = torch.cat([val_pos_pred, val_neg_pred])
            val_label = torch.cat([torch.ones(val_pos_pred.size(0)), torch.zeros(val_neg_pred.size(0))])

            # Use .numpy() if on CPU, or .cpu().numpy() if using GPU
            val_auc = roc_auc_score(val_label.detach().cpu().numpy(), torch.sigmoid(val_pred).detach().cpu().numpy())
            print(f'Epoch {epoch:03d} | Loss: {loss.item():.4f} | Val AUC: {val_auc:.4f}')

print("\nKYROFORM AI TRAINING COMPLETE!")

In [ ]:
import torch
from torch_geometric.data import HeteroData
import numpy as np

# 1. Ensure we only use proteins that have embeddings
valid_df = df_edges[df_edges['human'].isin(embeds) & df_edges['bacterial'].isin(embeds)].reset_index(drop=True)

# 2. Create strict unique lists and mappings
human_prots = sorted(valid_df['human'].unique().tolist())
bact_prots = sorted(valid_df['bacterial'].unique().tolist())

h2idx = {p: i for i, p in enumerate(human_prots)}
b2idx = {p: i for i, p in enumerate(bact_prots)}

# 3. Build the HeteroData object fresh
data = HeteroData()

# Node features: Ensuring order matches the mapping exactly
data['human'].x = torch.tensor(np.stack([embeds[p] for p in human_prots]), dtype=torch.float)
data['bacterial'].x = torch.tensor(np.stack([embeds[p] for p in bact_prots]), dtype=torch.float)

# Edge index: Mapping from the same valid_df
src = torch.tensor([h2idx[h] for h in valid_df['human']], dtype=torch.long)
dst = torch.tensor([b2idx[b] for b in valid_df['bacterial']], dtype=torch.long)

# Bi-directional edges (important for GNN message passing)
data['human', 'interacts', 'bacterial'].edge_index = torch.stack([src, dst])
data['bacterial', 'rev_interacts', 'human'].edge_index = torch.stack([dst, src])

print(f"Human nodes: {data['human'].x.size(0)} | Max Human Index: {src.max().item()}")
print(f"Bact nodes: {data['bacterial'].x.size(0)} | Max Bact Index: {dst.max().item()}")

NameError: name 'embeds' is not defined